
# JogjaPulse - Dataset Profile & Quality Check

**Purpose:** profile every file in `RawDataset/` and flag anything that needs cleaning or that affects integration - missing rows, wrong formats, out-of-range values, duplicates, and cross-file consistency.

**Important framing:** this is a *published, privacy-protected aggregate*. Many "missing rows" are **by design** (k=5 suppression; see README S5): a missing row means "fewer than 5 devices", **not zero, not data loss**. This notebook separates structural gaps (suppression + zero-device slots) from genuine format problems.

**How to use:** run top to bottom. Adjust `DATA_DIR` in the setup cell if your path differs. A `RESULTS` dict collects a verdict printed at the end.


In [1]:

import os, re
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Resolve data dir robustly (notebook lives in IndividualCode/1_ExploreCleaning/)
CANDIDATES = [
    r"d:\UGM Things\Semester 5\Data Engineering\Projects\DE1Assignment_JogjaPulse\RawDataset",
    os.path.abspath("../../RawDataset"),
    os.path.abspath("../RawDataset"),
    os.path.abspath("RawDataset"),
]
DATA_DIR = next((p for p in CANDIDATES if os.path.isdir(p)), CANDIDATES[0])
print("DATA_DIR =", DATA_DIR)

EXPECTED_DATES = pd.date_range("2021-12-01", "2022-04-30", freq="D")   # 151 days
EXPECTED_MONTHS = ["2021-12", "2022-01", "2022-02", "2022-03", "2022-04"]
EXPECTED_KAB = ["KULON PROGO", "BANTUL", "GUNUNGKIDUL", "SLEMAN", "YOGYAKARTA"]
N_KAPANEWON = 78

RESULTS = {}

def path(name):
    return os.path.join(DATA_DIR, name)

def profile_df(df, name):
    print(f"\n--- {name} ---")
    print("shape:", df.shape)
    print("dtypes:\n", df.dtypes.to_string())
    miss = df.isna().sum()
    miss = miss[miss > 0]
    print("missing NaN per col:", "none" if miss.empty else miss.to_dict())
    obj = df.select_dtypes(include=["object", "string"])
    empties = {c: int((df[c].astype(str).str.strip() == "").sum()) for c in obj.columns}
    empties = {c: n for c, n in empties.items() if n}
    print("empty-string cells:", "none" if not empties else empties)
    print("fully duplicate rows:", int(df.duplicated().sum()))


DATA_DIR = d:\UGM Things\Semester 5\Data Engineering\Projects\DE1Assignment_JogjaPulse\RawDataset



## 1. `presence_cell500m_daily.csv` - distinct devices per 500m grid cell per day

Watch for: date completeness (all 151 days), `cell_id` format `E{i}_N{j}` and its consistency with `x_min_utm`/`y_min_utm` (divisible by 500), `n_maids >= 5`, lon/lat inside DIY, and the size of the suppressed/zero-device gap vs the full grid. This file intentionally has **no `n_pings`** column.


In [2]:

df = pd.read_csv(path("presence_cell500m_daily.csv"))
profile_df(df, "presence_cell500m_daily.csv")

print("columns:", list(df.columns))
bad = df[~df["cell_id"].str.match(r"^E\d+_N\d+$")]
print("bad cell_id rows:", len(bad))
exp = "E" + (df["x_min_utm"] // 500).astype(int).astype(str) + "_N" + (df["y_min_utm"] // 500).astype(int).astype(str)
print("cell_id mismatch vs x_min/500,y_min/500:", int((exp != df["cell_id"]).sum()))
print("x not divisible by 500:", int((df["x_min_utm"] % 500 != 0).sum()), "| y not divisible by 500:", int((df["y_min_utm"] % 500 != 0).sum()))
d = pd.to_datetime(df["date"])
print("date range:", d.min().date(), "->", d.max().date(), "| distinct days:", d.nunique())
missing_days = sorted(set(EXPECTED_DATES.date) - set(d.dt.date))
print("missing calendar days:", missing_days if missing_days else "none")
print("distinct cells:", df["cell_id"].nunique())
print("n_maids: min", df["n_maids"].min(), "max", df["n_maids"].max(), "| <5:", int((df["n_maids"] < 5).sum()), "| <=0:", int((df["n_maids"] <= 0).sum()))
print("lon range:", df.centroid_lon.min(), df.centroid_lon.max(), "| lat range:", df.centroid_lat.min(), df.centroid_lat.max())
full = df["cell_id"].nunique() * d.nunique()
print(f"full grid {full} vs published {len(df)} -> missing {full - len(df)} ({(full - len(df)) / full * 100:.1f}%) [suppression + zero-device slots]")

RESULTS["presence_cell500m_daily"] = {
    "rows": int(len(df)),
    "distinct_days": int(d.nunique()),
    "missing_days": missing_days if missing_days else "none",
    "distinct_cells": int(df["cell_id"].nunique()),
    "bad_cell_id_rows": int(len(bad)),
    "cell_id_xy_mismatch": int((exp != df["cell_id"]).sum()),
    "n_maids_min": int(df["n_maids"].min()),
    "n_maids_below_5": int((df["n_maids"] < 5).sum()),
    "duplicate_rows": int(df.duplicated().sum()),
    "grid_gap_pct": round((full - len(df)) / full * 100, 1),
}



--- presence_cell500m_daily.csv ---
shape: (183548, 7)
dtypes:
 cell_id             str
x_min_utm         int64
y_min_utm         int64
centroid_lon    float64
centroid_lat    float64
date                str
n_maids           int64
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
columns: ['cell_id', 'x_min_utm', 'y_min_utm', 'centroid_lon', 'centroid_lat', 'date', 'n_maids']


bad cell_id rows: 0


cell_id mismatch vs x_min/500,y_min/500: 0
x not divisible by 500: 0 | y not divisible by 500: 0
date range: 2021-12-01 -> 2022-04-30 | distinct days: 151
missing calendar days: none
distinct cells: 5715
n_maids: min 5 max 10971 | <5: 0 | <=0: 0
lon range: 110.031602 110.816173 | lat range: -8.17589 -7.578602


full grid 862965 vs published 183548 -> missing 679417 (78.7%) [suppression + zero-device slots]



## 2. `presence_kapanewon_hourly.csv` - distinct devices + pings per kapanewon per hour

Watch for: 78 kapanewon, 5 kab_kota with expected names, date completeness, hours 0-23 all present, `n_maids >= 5`, and `n_pings >= n_maids` (presence requires >=1 ping). The gap vs the full 78 x 151 x 24 grid is suppression + zero-device slots.


In [3]:

df = pd.read_csv(path("presence_kapanewon_hourly.csv"), dtype={"kab_code": str})
profile_df(df, "presence_kapanewon_hourly.csv")

print("columns:", list(df.columns))
print("kab_kota set:", sorted(df.kab_kota.unique()))
print("unexpected kab_kota:", set(df.kab_kota) - set(EXPECTED_KAB))
print("distinct kapanewon:", df.kapanewon.nunique(), "| distinct kapanewon_code:", df.kapanewon_code.nunique())
name_collisions = df.groupby("kapanewon").kapanewon_code.nunique()
name_collisions = name_collisions[name_collisions > 1]
print("kapanewon names shared by >1 code:", name_collisions.to_dict() if len(name_collisions) else "none")
print("kab_code format bad:", int((~df.kab_code.str.match(r"^\d{2}\.\d{2}$")).sum()))
print("kapanewon_code format bad:", int((~df.kapanewon_code.str.match(r"^\d{2}\.\d{2}\.\d{2}$")).sum()))
d = pd.to_datetime(df["date"])
print("date range:", d.min().date(), "->", d.max().date(), "| distinct days:", d.nunique())
missing_days = sorted(set(EXPECTED_DATES.date) - set(d.dt.date))
print("missing calendar days:", missing_days if missing_days else "none")
print("hour set:", [int(x) for x in sorted(df.hour.unique())], "| out of 0-23 rows:", len(df[~df.hour.between(0, 23)]))
print("n_maids: min", df.n_maids.min(), "max", df.n_maids.max(), "| <5:", int((df.n_maids < 5).sum()))
print("n_pings: min", df.n_pings.min(), "| n_pings<n_maids:", int((df.n_pings < df.n_maids).sum()), "| n_pings<=0:", int((df.n_pings <= 0).sum()))
print("dup on (kapanewon_code,date,hour):", int(df.duplicated(["kapanewon_code", "date", "hour"]).sum()))
full = df.kapanewon_code.nunique() * d.nunique() * 24
print(f"full grid {full} vs published {len(df)} -> missing {full - len(df)} ({(full - len(df)) / full * 100:.1f}%) [suppression + zero-device slots]")

RESULTS["presence_kapanewon_hourly"] = {
    "rows": int(len(df)),
    "distinct_days": int(d.nunique()),
    "missing_days": missing_days if missing_days else "none",
    "distinct_kapanewon_names": int(df.kapanewon.nunique()),
    "distinct_kapanewon_codes": int(df.kapanewon_code.nunique()),
    "name_collisions": name_collisions.to_dict() if len(name_collisions) else "none",
    "kab_kota_ok": set(df.kab_kota) == set(EXPECTED_KAB),
    "hours_ok": [int(x) for x in sorted(df.hour.unique())] == list(range(24)),
    "n_maids_min": int(df.n_maids.min()),
    "n_pings_lt_n_maids": int((df.n_pings < df.n_maids).sum()),
    "duplicate_keys": int(df.duplicated(["kapanewon_code", "date", "hour"]).sum()),
    "grid_gap_pct": round((full - len(df)) / full * 100, 1),
}



--- presence_kapanewon_hourly.csv ---
shape: (230418, 8)
dtypes:
 kab_code            str
kab_kota            str
kapanewon_code      str
kapanewon           str
date                str
hour              int64
n_maids           int64
n_pings           int64
missing NaN per col: none


empty-string cells: none


fully duplicate rows: 0
columns: ['kab_code', 'kab_kota', 'kapanewon_code', 'kapanewon', 'date', 'hour', 'n_maids', 'n_pings']
kab_kota set: ['BANTUL', 'GUNUNGKIDUL', 'KULON PROGO', 'SLEMAN', 'YOGYAKARTA']
unexpected kab_kota: set()
distinct kapanewon: 77 | distinct kapanewon_code: 78
kapanewon names shared by >1 code: {'JETIS': 2}


kab_code format bad:

 0
kapanewon_code format bad: 0
date range: 2021-12-01 -> 2022-04-30 | distinct days: 151
missing calendar days: none
hour set: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23] | out of 0-23 rows: 0
n_maids: min 5 max 2506 | <5: 0
n_pings: min 5 | n_pings<n_maids: 0 | n_pings<=0: 0
dup on (kapanewon_code,date,hour): 0
full grid 282672 vs published 230418 -> missing 52254 (18.5%) [suppression + zero-device slots]



## 3. `home_maids_vs_population_kapanewon.csv` - devices by home kapanewon vs official population

Watch for: exactly 78 rows, unique kapanewon, no NaN/inf in `ratio`, `pop_actual > 0`, and `ratio == n_users / pop_actual`.


In [4]:

df = pd.read_csv(path("home_maids_vs_population_kapanewon.csv"))
profile_df(df, "home_maids_vs_population_kapanewon.csv")

print("rows == 78:", len(df) == 78, "(", len(df), ")")
print("kapanewon unique:", df.KEC_KAPN.nunique(), "| dup:", int(df.KEC_KAPN.duplicated().sum()))
print("ratio NaN:", int(df.ratio.isna().sum()), "| inf:", int(np.isinf(df.ratio).sum()))
print("pop_actual <=0:", int((df.pop_actual <= 0).sum()))
chk = np.isclose(df.ratio, df.n_users / df.pop_actual)
print("ratio mismatch vs n_users/pop_actual:", int((~chk).sum()))
print("ratio range:", df.ratio.min(), df.ratio.max())
print("has code column:", any("code" in c.lower() for c in df.columns))
dup_name = df[df.KEC_KAPN.duplicated(keep=False)].sort_values("KEC_KAPN")
print("duplicate-name rows:\n", dup_name[["KAB_KOTA", "KEC_KAPN", "n_users", "pop_actual"]].to_string())

RESULTS["home_maids_vs_population_kapanewon"] = {
    "rows": int(len(df)),
    "rows_eq_78": len(df) == 78,
    "kapanewon_unique": int(df.KEC_KAPN.nunique()),
    "kapanewon_dup": int(df.KEC_KAPN.duplicated().sum()),
    "ratio_nan": int(df.ratio.isna().sum()),
    "ratio_inf": int(np.isinf(df.ratio).sum()),
    "ratio_mismatch": int((~chk).sum()),
    "pop_le0": int((df.pop_actual <= 0).sum()),
}



--- home_maids_vs_population_kapanewon.csv ---
shape: (78, 5)
dtypes:
 KAB_KOTA          str
KEC_KAPN          str
n_users         int64
pop_actual    float64
ratio         float64
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
rows == 78: True ( 78 )
kapanewon unique: 77 | dup: 1
ratio NaN: 0 | inf: 0
pop_actual <=0: 0
ratio mismatch vs n_users/pop_actual: 0
ratio range: 0.0093590470788429 2.247003891050584
has code column: False
duplicate-name rows:
       KAB_KOTA KEC_KAPN  n_users  pop_actual
4   YOGYAKARTA    JETIS     6922     23480.0
27      BANTUL    JETIS     7403     59040.0



## 4. `validation_night_presence_kapanewon.csv` - night-time presence (validation)

Watch for: 156 rows = 78 kapanewon x 2 periods, both expected periods present, `slots_published_pct` in 0-100.


In [5]:

df = pd.read_csv(path("validation_night_presence_kapanewon.csv"))
profile_df(df, "validation_night_presence_kapanewon.csv")

print("rows:", len(df), "| periods:", sorted(df.period.unique()))
print("kapanewon count per period:\n", df.groupby("period").kapanewon.nunique())
print("night_presence NaN:", int(df.night_presence.isna().sum()), "| slots_published_pct range:", df.slots_published_pct.min(), df.slots_published_pct.max())
print("kapanewon total unique:", df.kapanewon.nunique())
print("has kapanewon_code column:", "kapanewon_code" in df.columns)
print("dup (period, kapanewon):", int(df.duplicated(["period", "kapanewon"]).sum()))
print("dup (period, kab_kota, kapanewon):", int(df.duplicated(["period", "kab_kota", "kapanewon"]).sum()))
print("rows per period:\n", df.groupby("period").size().to_string())
dup_rows = df[df.duplicated(["period", "kapanewon"], keep=False)].sort_values(["period", "kapanewon"])
print("duplicate-name rows:\n", dup_rows[["period", "kab_kota", "kapanewon", "n_users", "pop_actual"]].to_string())

RESULTS["validation_night_presence_kapanewon"] = {
    "rows": int(len(df)),
    "rows_eq_156": len(df) == 156,
    "periods": sorted(df.period.unique()),
    "kapanewon_total": int(df.kapanewon.nunique()),
    "night_presence_nan": int(df.night_presence.isna().sum()),
    "slots_pct_range": [float(df.slots_published_pct.min()), float(df.slots_published_pct.max())],
}



--- validation_night_presence_kapanewon.csv ---
shape: (156, 7)
dtypes:
 period                     str
kab_kota                   str
kapanewon                  str
night_presence         float64
slots_published_pct    float64
n_users                  int64
pop_actual             float64
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
rows: 156 | periods: ['Dec 2021 - Apr 2022', 'Dec 2021 - Jan 2022']
kapanewon count per period:
 period
Dec 2021 - Apr 2022    77
Dec 2021 - Jan 2022    77
Name: kapanewon, dtype: int64
night_presence NaN: 0 | slots_published_pct range: 3.2008830022075054 100.0
kapanewon total unique: 77
has kapanewon_code column: False
dup (period, kapanewon): 2
dup (period, kab_kota, kapanewon): 0
rows per period:
 period
Dec 2021 - Apr 2022    78
Dec 2021 - Jan 2022    78
duplicate-name rows:
                   period    kab_kota kapanewon  n_users  pop_actual
5    Dec 2021 - Apr 2022      BANTUL     JETIS     7403     59040.0
68   Dec 2021


## 5. `tier1_summary.csv` - suppression statistics for the two presence products

Watch for: 2 rows, products `A_cell500m_daily` and `B_kapanewon_hourly`, and that `rows_published` matches the *actual* row counts of files 1 and 2 (cross-file check).


In [6]:

df = pd.read_csv(path("tier1_summary.csv"))
profile_df(df, "tier1_summary.csv")
print(df.to_string())
print("products:", list(df["product"]))
cell = pd.read_csv(path("presence_cell500m_daily.csv"), usecols=[0])
kap = pd.read_csv(path("presence_kapanewon_hourly.csv"), usecols=[0])
a = int(df.loc[df["product"] == "A_cell500m_daily", "rows_published"].iloc[0])
b = int(df.loc[df["product"] == "B_kapanewon_hourly", "rows_published"].iloc[0])
print(f"A rows_published {a} vs actual {len(cell)} -> {'OK' if a == len(cell) else 'MISMATCH'}")
print(f"B rows_published {b} vs actual {len(kap)} -> {'OK' if b == len(kap) else 'MISMATCH'}")

RESULTS["tier1_summary"] = {
    "rows": int(len(df)),
    "products": list(df["product"]),
    "A_rows_published_vs_actual": "OK" if a == len(cell) else f"MISMATCH ({a} vs {len(cell)})",
    "B_rows_published_vs_actual": "OK" if b == len(kap) else f"MISMATCH ({b} vs {len(kap)})",
}



--- tier1_summary.csv ---
shape: (2, 8)
dtypes:
 pings_inside_diy_pct            float64
product                             str
k                                 int64
rows_total                        int64
rows_published                    int64
rows_suppressed_pct             float64
maid_presence_suppressed_pct    float64
n_cells_published                 int64
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
   pings_inside_diy_pct             product  k  rows_total  rows_published  rows_suppressed_pct  maid_presence_suppressed_pct  n_cells_published
0                 100.0    A_cell500m_daily  5      518176          183548                 64.6                           6.6               5715
1                 100.0  B_kapanewon_hourly  5      273979          230418                 15.9                           0.7                 78
products: ['A_cell500m_daily', 'B_kapanewon_hourly']
A rows_published 183548 vs actual 183548 -> OK
B rows_published 230


## 6. `od_home_visits_kapanewon_monthly.csv` - home-based visitation flows (monthly)

Watch for: 5 months, `reduced_quality=True` only for 2022-03 and 2022-04, code format `dd.dd.dd`, `n_maids >= 5`, unique (month, origin, dest) keys, and that intra-zonal rows (origin == dest) are present.


In [7]:

df = pd.read_csv(path("od_home_visits_kapanewon_monthly.csv"), dtype={"origin_kab_code": str, "dest_kab_code": str})
profile_df(df, "od_home_visits_kapanewon_monthly.csv")

print("months:", sorted(df.month.unique()))
print("missing months:", set(EXPECTED_MONTHS) - set(df.month.unique()) or "none")
print("month format bad:", int((~df.month.str.match(r"^\d{4}-\d{2}$")).sum()))
print("origin_code format bad:", int((~df.origin_code.str.match(r"^\d{2}\.\d{2}\.\d{2}$")).sum()))
print("dest_code format bad:", int((~df.dest_code.str.match(r"^\d{2}\.\d{2}\.\d{2}$")).sum()))
print("distinct origin:", df.origin_code.nunique(), "| distinct dest:", df.dest_code.nunique())
print("reduced_quality True months:", sorted(df.loc[df.reduced_quality, "month"].unique()))
print("reduced_quality False months:", sorted(df.loc[~df.reduced_quality, "month"].unique()))
print("n_maids: min", df.n_maids.min(), "| <5:", int((df.n_maids < 5).sum()))
print("dup on (month,origin_code,dest_code):", int(df.duplicated(["month", "origin_code", "dest_code"]).sum()))
print("intra-zonal (origin==dest) rows:", int((df.origin_code == df.dest_code).sum()))
print("reduced_quality dtype:", df.reduced_quality.dtype, "| values:", df.reduced_quality.unique())

RESULTS["od_home_visits_kapanewon_monthly"] = {
    "rows": int(len(df)),
    "months": sorted(df.month.unique()),
    "missing_months": list(set(EXPECTED_MONTHS) - set(df.month.unique())),
    "distinct_origin": int(df.origin_code.nunique()),
    "distinct_dest": int(df.dest_code.nunique()),
    "reduced_quality_months": sorted(df.loc[df.reduced_quality, "month"].unique()),
    "n_maids_min": int(df.n_maids.min()),
    "duplicate_keys": int(df.duplicated(["month", "origin_code", "dest_code"]).sum()),
    "intra_zonal_rows": int((df.origin_code == df.dest_code).sum()),
}



--- od_home_visits_kapanewon_monthly.csv ---
shape: (9894, 11)
dtypes:
 month                 str
origin_kab_code       str
origin_kab_kota       str
origin_code           str
origin_kapanewon      str
dest_kab_code         str
dest_kab_kota         str
dest_code             str
dest_kapanewon        str
n_maids             int64
reduced_quality      bool
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
months: ['2021-12', '2022-01', '2022-02', '2022-03', '2022-04']
missing months: none
month format bad: 0
origin_code format bad: 0
dest_code format bad: 0
distinct origin: 78 | distinct dest: 78
reduced_quality True months: ['2022-03', '2022-04']
reduced_quality False months: ['2021-12', '2022-01', '2022-02']
n_maids: min 5 | <5: 0
dup on (month,origin_code,dest_code): 0
intra-zonal (origin==dest) rows: 390
reduced_quality dtype: bool | values: [False  True]



## 7. `od_origin_homes_monthly.csv` - devices homed per origin kapanewon per month (OD denominators)

Watch for: 390 rows = 78 kapanewon x 5 months, `reduced_quality` flag consistent with file 6, and `n_homed_maids_daytime` NaN where the count is below 5 (by design).


In [8]:

df = pd.read_csv(path("od_origin_homes_monthly.csv"), dtype={"origin_kab_code": str})
profile_df(df, "od_origin_homes_monthly.csv")

print("months:", sorted(df.month.unique()))
print("rows:", len(df), "| expected 78*5=390 ->", len(df) == 390)
print("origin kapanewon per month:\n", df.groupby("month").origin_code.nunique())
print("n_homed_maids <5:", int((df.n_homed_maids < 5).sum()))
print("n_homed_maids_daytime NaN (expected when <5):", int(df.n_homed_maids_daytime.isna().sum()))
print("reduced_quality True months:", sorted(df.loc[df.reduced_quality, "month"].unique()))

RESULTS["od_origin_homes_monthly"] = {
    "rows": int(len(df)),
    "rows_eq_390": len(df) == 390,
    "months": sorted(df.month.unique()),
    "origin_per_month": df.groupby("month").origin_code.nunique().to_dict(),
    "n_homed_maids_below_5": int((df.n_homed_maids < 5).sum()),
    "daytime_nan": int(df.n_homed_maids_daytime.isna().sum()),
    "reduced_quality_months": sorted(df.loc[df.reduced_quality, "month"].unique()),
}



--- od_origin_homes_monthly.csv ---
shape: (390, 8)
dtypes:
 month                      str
origin_kab_code            str
origin_kab_kota            str
origin_code                str
origin_kapanewon           str
n_homed_maids            int64
n_homed_maids_daytime    int64
reduced_quality           bool
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
months: ['2021-12', '2022-01', '2022-02', '2022-03', '2022-04']
rows: 390 | expected 78*5=390 -> True
origin kapanewon per month:
 month
2021-12    78
2022-01    78
2022-02    78
2022-03    78
2022-04    78
Name: origin_code, dtype: int64
n_homed_maids <5: 0
n_homed_maids_daytime NaN (expected when <5): 0
reduced_quality True months: ['2022-03', '2022-04']



## 8. `od_summary.csv` - suppression statistics for the OD product

Watch for: 1 row, `k=5`, `min_nights=2`, and that `od_rows_published` / `origin_rows_published` match the actual row counts of files 6 and 7.


In [9]:

df = pd.read_csv(path("od_summary.csv"))
profile_df(df, "od_summary.csv")
print(df.to_string())
odv = pd.read_csv(path("od_home_visits_kapanewon_monthly.csv"), usecols=[0])
odh = pd.read_csv(path("od_origin_homes_monthly.csv"), usecols=[0])
op = int(df.od_rows_published.iloc[0])
orp = int(df.origin_rows_published.iloc[0])
print(f"od_rows_published {op} vs actual od_home_visits {len(odv)} -> {'OK' if op == len(odv) else 'MISMATCH'}")
print(f"origin_rows_published {orp} vs actual od_origin_homes {len(odh)} -> {'OK' if orp == len(odh) else 'MISMATCH'}")

RESULTS["od_summary"] = {
    "rows": int(len(df)),
    "k": int(df.k.iloc[0]),
    "min_nights": int(df.min_nights.iloc[0]),
    "od_rows_published_vs_actual": "OK" if op == len(odv) else f"MISMATCH ({op} vs {len(odv)})",
    "origin_rows_published_vs_actual": "OK" if orp == len(odh) else f"MISMATCH ({orp} vs {len(odh)})",
}



--- od_summary.csv ---
shape: (1, 7)
dtypes:
 k                              int64
min_nights                     int64
od_rows_total                  int64
od_rows_published              int64
od_rows_suppressed_pct       float64
inter_zonal_flow_kept_pct    float64
origin_rows_published          int64
missing NaN per col: none
empty-string cells: none
fully duplicate rows: 0
   k  min_nights  od_rows_total  od_rows_published  od_rows_suppressed_pct  inter_zonal_flow_kept_pct  origin_rows_published
0  5           2          19263               9894                    48.6                       95.3                    390
od_rows_published 9894 vs actual od_home_visits 9894 -> OK
origin_rows_published 390 vs actual od_origin_homes 390 -> OK



## 9. Cross-file consistency

There are **78 kapanewon codes** but only **77 kapanewon names**, because **JETIS** is shared by two codes (one in Bantul, one in Yogyakarta). So compare by **code** where available; joining by **name** alone will collide the two JETIS rows. Only `presence_kapanewon_hourly` and the OD files carry `kapanewon_code`; `home_maids_vs_population` and `validation_night_presence` have names only, so disambiguate them with `kab_kota`.


In [10]:

pk = pd.read_csv(path("presence_kapanewon_hourly.csv"), usecols=["kapanewon", "kapanewon_code"])
hm = pd.read_csv(path("home_maids_vs_population_kapanewon.csv"), usecols=["KEC_KAPN"])
vn = pd.read_csv(path("validation_night_presence_kapanewon.csv"), usecols=["kapanewon"])
odv = pd.read_csv(path("od_home_visits_kapanewon_monthly.csv"), usecols=["origin_kapanewon", "dest_kapanewon", "origin_code", "dest_code"])
odh = pd.read_csv(path("od_origin_homes_monthly.csv"), usecols=["origin_kapanewon", "origin_code"])

# by NAME (every file has a name col; JETIS is shared by 2 codes -> 77 unique names)
set_pk = set(pk.kapanewon); set_hm = set(hm.KEC_KAPN); set_vn = set(vn.kapanewon)
set_od = set(odv.origin_kapanewon) | set(odv.dest_kapanewon) | set(odh.origin_kapanewon)
print("[by name] presence", len(set_pk), "| home", len(set_hm), "| validation", len(set_vn), "| od", len(set_od))
print("[by name] all match:", set_pk == set_hm == set_vn == set_od)

# by CODE (only presence + OD carry codes)
set_pk_code = set(pk.kapanewon_code)
set_od_code = set(odv.origin_code) | set(odv.dest_code) | set(odh.origin_code)
print("[by code] presence", len(set_pk_code), "| od", len(set_od_code), "| match:", set_pk_code == set_od_code)
print("home has code col:", any("code" in c.lower() for c in hm.columns), "| validation has code col:", any("code" in c.lower() for c in vn.columns))
print("=> safe join key is kapanewon_code (78). home & validation have name only -> disambiguate JETIS with kab_kota.")

RESULTS["_kapanewon_consistency"] = {
    "by_name": {"presence": len(set_pk), "home": len(set_hm), "validation": len(set_vn), "od": len(set_od), "all_match": set_pk == set_hm == set_vn == set_od},
    "by_code": {"presence": len(set_pk_code), "od": len(set_od_code), "match": set_pk_code == set_od_code},
    "home_has_code": False, "validation_has_code": False,
    "name_collision": "JETIS (2 codes share the name)",
}


[by name] presence

 77 | home 77 | validation 77 | od 77
[by name] all match: True
[by code] presence 78 | od 78 | match: True
home has code col: False | validation has code col: False
=> safe join key is kapanewon_code (78). home & validation have name only -> disambiguate JETIS with kab_kota.



## 10. Verdict summary

Compiled from the `RESULTS` dict collected above. Read this first for a quick "is it clean?" answer; the per-file cells above have the details.


In [11]:

print("=" * 72)
print("VERDICT SUMMARY")
print("=" * 72)
for name, info in RESULTS.items():
    print(f"\n## {name}")
    for k, v in info.items():
        print(f"   {k}: {v}")


VERDICT SUMMARY

## presence_cell500m_daily
   rows: 183548
   distinct_days: 151
   missing_days: none
   distinct_cells: 5715
   bad_cell_id_rows: 0
   cell_id_xy_mismatch: 0
   n_maids_min: 5
   n_maids_below_5: 0
   duplicate_rows: 0
   grid_gap_pct: 78.7

## presence_kapanewon_hourly
   rows: 230418
   distinct_days: 151
   missing_days: none
   distinct_kapanewon_names: 77
   distinct_kapanewon_codes: 78
   name_collisions: {'JETIS': 2}
   kab_kota_ok: True
   hours_ok: True
   n_maids_min: 5
   n_pings_lt_n_maids: 0
   duplicate_keys: 0
   grid_gap_pct: 18.5

## home_maids_vs_population_kapanewon
   rows: 78
   rows_eq_78: True
   kapanewon_unique: 77
   kapanewon_dup: 1
   ratio_nan: 0
   ratio_inf: 0
   ratio_mismatch: 0
   pop_le0: 0

## validation_night_presence_kapanewon
   rows: 156
   rows_eq_156: True
   periods: ['Dec 2021 - Apr 2022', 'Dec 2021 - Jan 2022']
   kapanewon_total: 77
   night_presence_nan: 0
   slots_pct_range: [3.2008830022075054, 100.0]

## tier1_summary